# VMOL Protocol — GRPO training for Lending Risk Governor

Trains **Qwen 2.5 1.5B Instruct** via GRPO to act as an AI risk governor that adjusts LTV and Liquidation Threshold on a mock Aave-style lending pool.

- Code: https://github.com/platanus-build-night/platanus-build-night-26-ve-henryf10h
- Target: T4 GPU (Colab free tier)
- ETA: ~30min for 50 steps smoke / ~3h for 500 steps

**Pipeline:** `scenarios.py` (ETH paths) → `prompt.py` → Qwen 2.5 1.5B → JSON → `parse_output` → `reward.py` → GRPO update.

## 1 · Environment check

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"bf16 supported: {torch.cuda.is_bf16_supported()}")

## 2 · Install dependencies
Unsloth + TRL for GRPO.

In [ ]:
%%capture
!pip install --upgrade pip
!pip install unsloth vllm
!pip install --upgrade --no-cache-dir --no-deps git+https://github.com/huggingface/trl.git

## 3 · Clone the VMOL repo
Brings `config.py`, `sim.py`, `scenarios.py`, `prompt.py`, `reward.py` into the Colab FS.

In [ ]:
import os, sys, subprocess

REPO_DIR = "/content/vmol"
if not os.path.isdir(REPO_DIR):
    subprocess.run([
        "git", "clone",
        "https://github.com/platanus-build-night/platanus-build-night-26-ve-henryf10h.git",
        REPO_DIR,
    ], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

sys.path.insert(0, f"{REPO_DIR}/rl")
os.chdir(f"{REPO_DIR}/rl")
!ls

## 4 · Imports — Unsloth + our modules

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported
import torch
import copy
import numpy as np
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer

from config import TRAIN, SIM, GUARD, POOL
from sim import LendingSimulator
from scenarios import generate_batch
from prompt import build_prompt, parse_output, ParseError
from reward import compute_reward

print(f"Base model: {TRAIN.base_model}")
print(f"LoRA rank:  {TRAIN.lora_rank}")
print(f"Max steps:  {TRAIN.max_steps}")
print(f"KL coef:    {TRAIN.kl_coef}")
print(f"bf16:       {is_bfloat16_supported()}")

## 5 · Load Qwen 2.5 1.5B Instruct with LoRA

In [ ]:
MAX_SEQ_LEN = TRAIN.max_prompt_length + TRAIN.max_completion_length

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=TRAIN.base_model,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=False,
    fast_inference=False,
    max_lora_rank=TRAIN.lora_rank,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=TRAIN.lora_rank,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=TRAIN.lora_rank,
    use_gradient_checkpointing="unsloth",
    random_state=TRAIN.seed,
)
print("Model + LoRA ready.")

## 6 · Build the training dataset
Pre-generate N scenarios (ETH paths), pre-run each lending sim to a non-trivial state, snapshot (prompt, simulator). At train time GRPO generates G completions per prompt; the reward fn deep-copies the matching simulator for each completion.

In [ ]:
N_TRAIN_SCENARIOS = 256
PRE_ROLL_MIN, PRE_ROLL_MAX = 10, 80

scenarios_pool = generate_batch(n=N_TRAIN_SCENARIOS, seed=TRAIN.seed)

sim_pool = []
prompt_messages_pool = []
rng = np.random.default_rng(seed=TRAIN.seed + 1)

for idx, scenario in enumerate(scenarios_pool):
    sim = LendingSimulator(
        eth_path=scenario.eth_path,
        initial_ltv=scenario.initial_ltv,
        initial_liq_threshold=scenario.initial_liq_threshold,
        rng=np.random.default_rng(seed=TRAIN.seed + idx),
    )
    pre_roll = int(rng.integers(PRE_ROLL_MIN, PRE_ROLL_MAX))
    history = sim.run_forward(n_steps=pre_roll)
    state = sim.get_state()
    hf_hist = [h.avg_health_factor for h in history[-8:]]
    sys_p, user_p = build_prompt(
        state=state,
        eth_history=scenario.eth_path[: sim.step_idx + 1],
        health_factor_history=hf_hist,
    )
    sim_pool.append(sim)
    prompt_messages_pool.append([
        {"role": "system", "content": sys_p},
        {"role": "user",   "content": user_p},
    ])

dataset = Dataset.from_list([
    {"prompt": prompt_messages_pool[i], "scenario_idx": i}
    for i in range(len(scenarios_pool))
])
print(dataset)
print("--- first prompt (truncated) ---")
print(prompt_messages_pool[0][1]["content"][:500])

## 7 · Reward function + parse-success diagnostic

In [ ]:
def _completion_text(c):
    if isinstance(c, str):
        return c
    if isinstance(c, list) and c and isinstance(c[0], dict):
        return "\n".join(str(m.get("content", "")) for m in c)
    return str(c)


def lending_reward(prompts, completions, scenario_idx, **kwargs):
    rewards = []
    for completion, idx in zip(completions, scenario_idx):
        text = _completion_text(completion)
        sim = copy.deepcopy(sim_pool[idx])
        br = compute_reward(text, sim)
        rewards.append(float(br.total))
    return rewards


def json_validity(prompts, completions, **kwargs):
    rewards = []
    for completion in completions:
        text = _completion_text(completion)
        parsed = parse_output(text)
        rewards.append(0.5 if not isinstance(parsed, ParseError) else -0.5)
    return rewards


sample = '{"action":"adjust","new_ltv":0.72,"new_liq_threshold":0.78,"is_emergency":false,"reasoning":"test"}'
print("lending_reward sample:", lending_reward([None], [sample], [0]))
print("json_validity sample: ", json_validity([None], [sample]))

## 8 · GRPO training config

In [ ]:
training_args = GRPOConfig(
    output_dir="outputs",
    use_vllm=False,
    learning_rate=TRAIN.learning_rate,
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    warmup_ratio=TRAIN.warmup_ratio,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    logging_steps=5,
    bf16=is_bfloat16_supported(),
    fp16=not is_bfloat16_supported(),
    per_device_train_batch_size=TRAIN.batch_size,
    gradient_accumulation_steps=1,
    num_generations=TRAIN.num_generations,
    max_steps=TRAIN.max_steps,
    save_steps=50,
    max_grad_norm=0.1,
    beta=TRAIN.kl_coef,
    seed=TRAIN.seed,
    report_to="none",
)
print(training_args)

## 9 · Create trainer + train
`save_steps=50` → checkpoint every 50 updates.

In [ ]:
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[lending_reward, json_validity],
    args=training_args,
    train_dataset=dataset,
)
trainer.train()

## 10 · Save LoRA adapters

In [ ]:
model.save_pretrained("vmol_lending_lora_v1")
tokenizer.save_pretrained("vmol_lending_lora_v1")
!ls -la vmol_lending_lora_v1

# Zip for easy download from Colab files panel
!zip -r vmol_lending_lora_v1.zip vmol_lending_lora_v1
print("\nDownload vmol_lending_lora_v1.zip from the Files tab on the left.")

## 11 · Qualitative test — agent on 3 scenarios
Sanity check: does the trained model respond differently to crash / pump / stable scenarios?

In [ ]:
from scenarios import _gen_crash, _gen_pump, _gen_stable

FastLanguageModel.for_inference(model)

peek_rng = np.random.default_rng(seed=777)
for name, gen in [("crash", _gen_crash), ("pump", _gen_pump), ("stable", _gen_stable)]:
    sc = gen(peek_rng, SIM.episode_steps)
    sim = LendingSimulator(
        eth_path=sc.eth_path,
        initial_ltv=sc.initial_ltv,
        initial_liq_threshold=sc.initial_liq_threshold,
        rng=np.random.default_rng(seed=7),
    )
    hist = sim.run_forward(n_steps=50)
    hf_hist = [h.avg_health_factor for h in hist[-8:]]
    state = sim.get_state()
    sys_p, user_p = build_prompt(
        state=state,
        eth_history=sc.eth_path[: sim.step_idx + 1],
        health_factor_history=hf_hist,
    )
    messages = [
        {"role": "system", "content": sys_p},
        {"role": "user",   "content": user_p},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    gen = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    parsed = parse_output(gen)
    print(f"\n=== {name} ===")
    print(f"  eth: {sc.eth_path[sim.step_idx - 5]:.0f} -> {sc.eth_path[sim.step_idx]:.0f}")
    print(f"  HF avg: {state.avg_health_factor:.3f}  min: {state.min_health_factor:.3f}")
    print(f"  Current LTV: {state.ltv*100:.1f}%  LT: {state.liq_threshold*100:.1f}%")
    print(f"  Model raw: {gen[:300]}")
    print(f"  Parsed:    {parsed}")